In [ ]:
import torch
from torch import nn
import torch.nn.init as init
import pandas as pd
import numpy as np
import re
from collections import Counter
from tqdm.auto import tqdm

In [ ]:
class S3(nn.Module):
    def __init__(self, num_layers, initial_num_segments, shuffle_vector_dim=1, segment_multiplier=1, segments_per_layer=None, use_conv_w_avg=True, initialization_type="kaiming", use_stitch=True):
        super().__init__()
        self.S3_layers = nn.ModuleList()
        next_segment_num = initial_num_segments
        for i in range(num_layers):
            self.S3_layers.append(S3Layer(next_segment_num, shuffle_vector_dim, use_conv_w_avg, initialization_type, use_stitch))
            next_segment_num = max(1, int(next_segment_num * segment_multiplier))

    def forward(self, x):
        x_copy = x.clone()
        for layer in self.S3_layers:
            if x_copy.shape[1] >= layer.num_segments:
                remainder = x_copy.shape[1] % layer.num_segments
                if remainder:
                    x_copy = x_copy[:, remainder:, :]
                x_copy = layer(x_copy)
        if x.shape[1] > x_copy.shape[1]:
            x_copy = torch.cat([x[:, :x.shape[1] - x_copy.shape[1], :], x_copy], dim=1)
        return x_copy

class S3Layer(nn.Module):
    def __init__(self, num_segments, shuffle_vector_dim=1, use_conv_w_avg=True, initialization_type="kaiming", use_stitch=True):
        super().__init__()
        self.num_segments = int(num_segments)
        self.use_conv_w_avg = use_conv_w_avg
        self.use_stitch = use_stitch
        self.shuffle_vector = nn.Parameter(torch.empty((self.num_segments,) * shuffle_vector_dim))
        if initialization_type == "kaiming" and shuffle_vector_dim > 1:
            init.kaiming_normal_(self.shuffle_vector, mode="fan_out", nonlinearity="relu")
            with torch.no_grad():
                self.shuffle_vector.mul_(0.001).add_(0.01)
        elif initialization_type == "manual" or shuffle_vector_dim == 1:
            with torch.no_grad():
                self.shuffle_vector.fill_(0.6)
        else:
            raise ValueError(initialization_type)
        if use_conv_w_avg:
            self.w_avg = nn.Conv1d(2, 1, 1)
        else:
            self.weights = nn.Parameter(torch.ones(2) * 0.5)

    def forward(self, x):
        scores = self.shuffle_vector.sum(tuple(range(self.shuffle_vector.ndim - 1))) if self.shuffle_vector.ndim > 1 else self.shuffle_vector
        order = torch.argsort(scores, descending=True)
        segments = torch.chunk(x, self.num_segments, dim=1)
        shuffled = torch.cat([segments[i] for i in order.tolist()], dim=1)
        if not self.use_stitch:
            return shuffled
        if self.use_conv_w_avg:
            b, t, c = x.shape
            return self.w_avg(torch.stack((shuffled, x), dim=-1).reshape(b * t * c, 2, 1)).reshape(b, t, c)
        weights = torch.softmax(self.weights, dim=0)
        return torch.stack((shuffled, x), dim=-1).mul(weights).sum(dim=-1)

In [ ]:
df_users = pd.read_csv("personalized_test.tsv", sep="\t")
df_news = pd.read_csv("news_min (2).tsv", sep="\t")
valid_news_ids = set(df_news["News ID"].dropna().astype(str).str.strip())
df_users["ClicknewsID_original"] = df_users["clicknewsID"].copy()

In [ ]:
s3_model = S3(num_layers=20, initial_num_segments=5, shuffle_vector_dim=3, use_conv_w_avg=True, initialization_type="kaiming", use_stitch=True)

In [ ]:
def normalize_ids(seq):
    if pd.isna(seq):
        return []
    return [x for x in str(seq).replace(",", " ").split() if x]

def apply_s3_augmentation(clicknews_ids, model, valid_ids):
    ids = list(clicknews_ids)
    if not ids or any(x not in valid_ids for x in ids):
        return ids
    for layer in model.S3_layers:
        if len(ids) < layer.num_segments:
            continue
        segments = [part.tolist() for part in np.array_split(np.asarray(ids, dtype=object), layer.num_segments)]
        with torch.no_grad():
            scores = layer.shuffle_vector
            if scores.ndim > 1:
                scores = scores.sum(tuple(range(scores.ndim - 1)))
            order = torch.argsort(scores, descending=True).tolist()
        ids = [item for i in order for item in segments[i]]
    return ids

def force_shuffle(ids, strength=0.35):
    ids = list(ids)
    if len(ids) < 2 or len(set(ids)) < 2:
        return ids
    k = min(len(ids), max(2, int(len(ids) * strength)))
    positions = np.random.choice(len(ids), k, replace=False)
    values = [ids[i] for i in positions]
    for _ in range(20):
        permuted = np.random.permutation(values).tolist()
        if permuted != values:
            break
    for i, value in zip(positions, permuted):
        ids[i] = value
    if ids == list(ids) and permuted == values:
        i = next(i for i in range(1, len(ids)) if ids[i] != ids[0])
        ids[0], ids[i] = ids[i], ids[0]
    return ids

In [ ]:
np.random.seed(42)
torch.manual_seed(42)
augmented_clicks = []
for _, row in tqdm(df_users.iterrows(), total=len(df_users), desc="Applying segment shuffle"):
    ids = normalize_ids(row["clicknewsID"])
    with torch.no_grad():
        for layer in s3_model.S3_layers:
            layer.shuffle_vector.copy_(torch.rand_like(layer.shuffle_vector))
    augmented = apply_s3_augmentation(ids, s3_model, valid_news_ids)
    if augmented == ids and all(x in valid_news_ids for x in ids):
        augmented = force_shuffle(augmented)
    augmented_clicks.append(" ".join(augmented))
df_users["clicknewsID"] = augmented_clicks

In [ ]:
df_users["actually_changed"] = [normalize_ids(a) != normalize_ids(b) for a, b in zip(df_users["ClicknewsID_original"], df_users["clicknewsID"])]
df_users["same_multiset"] = [Counter(normalize_ids(a)) == Counter(normalize_ids(b)) for a, b in zip(df_users["ClicknewsID_original"], df_users["clicknewsID"])]
df_users["all_valid"] = [all(x in valid_news_ids for x in normalize_ids(b)) for b in df_users["clicknewsID"]]
print(f"Changed rows: {df_users['actually_changed'].sum()} / {len(df_users)}")
print(f"Multiset-preserving rows: {df_users['same_multiset'].sum()} / {len(df_users)}")
print(f"Catalog-valid rows: {df_users['all_valid'].sum()} / {len(df_users)}")
df_users.loc[:10, ["userid", "ClicknewsID_original", "clicknewsID"]]

In [ ]:
df_users.to_csv("S3_on_test_5_segments.csv", index=False)